Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/monitor-models/data-drift/drift-on-aks.png)

# Monitor data drift on models deployed to Azure Kubernetes Service 

In this tutorial, you will setup a data drift monitor on a toy model that predicts elevation based on a few weather factors which will send email alerts if drift is detected.

## Prerequisites
If you are using an Azure Machine Learning Compute instance, you are all set. Otherwise, go through the [configuration notebook](../../../configuration.ipynb) first if you haven't already established your connection to the AzureML Workspace.

In [1]:
# Check core SDK version number
import azureml.core

print('SDK version:', azureml.core.VERSION)

SDK version: 0.1.0.6188706


## Initialize Workspace

Initialize a workspace object from persisted configuration.

In [2]:
from azureml.core import Workspace

ws = Workspace.from_config()
ws

Workspace.create(name='notebooks', subscription_id='b0d50d34-4247-4cae-871a-6b1799907b8d', resource_group='aml_git_nb_6188706')

## Setup training dataset and model

Setup the training dataset and model in preparation for deployment to the Azure Kubernetes Service. 

The next few cells will:
  * get the default datastore and upload the `training.csv` dataset to the datastore
  * create, register, and get the dataset
  * register the dataset
  * create the baseline as a time slice of the target dataset
  * optionally, register the baseline dataset
  
See the `setup.py` script in this folder for details on how `training.csv` and `elevation-regression-model.pkl` are created. 

In [3]:
# use default datastore
dstore = ws.get_default_datastore()

# upload weather data
dstore.upload('training-dataset', 'drift-on-aks-data', overwrite=True, show_progress=False);

In [4]:
# import Dataset class
from azureml.core import Dataset

# create dataset 
dset = Dataset.Tabular.from_delimited_files(dstore.path('drift-on-aks-data/training.csv'))
dset = dset.register(ws, 'drift-demo-dataset')
dset = Dataset.get_by_name(ws, 'drift-demo-dataset')

In [5]:
from azureml.core.model import Model

model = Model.register(model_path='elevation-regression-model.pkl',
                       model_name='elevation-regression-model.pkl',
                       tags={'area': "weather", 'type': "linear regression"},
                       description='Linear regression model to predict elevation based on the weather',
                       workspace=ws,
                       datasets=[(Dataset.Scenario.TRAINING, dset)])

Registering model elevation-regression-model.pkl


## Create the infernece config

Create the environment and inference config from the `myenv.yml` and `score.py` files. Notice the [Model Data Collector](https://docs.microsoft.com/azure/machine-learning/service/how-to-enable-data-collection) code included in the scoring script. This dependency is currently required to collect model data, but will be removed in the near future as data collection in Azure Machine Learning webservice endpoints is automated. 

In [6]:
from azureml.core import Environment

env = Environment.from_conda_specification(name='deploytocloudenv', file_path='myenv.yml')

In [7]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(entry_script="score.py", environment=env)

## Create the AKS compute target

In [8]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'drift-aks'
# Create the cluster
try:
    aks_target = ws.compute_targets[aks_name]
except:
    aks_target = ComputeTarget.create(workspace = ws,
                                      name = aks_name,
                                      provisioning_configuration = prov_config)

    # Wait for the create process to complete
    aks_target.wait_for_completion(show_output = True)

Creating....................................................................................................................................................................................
SucceededProvisioning operation finished, operation "Succeeded"


## Deploy the model to AKS 

Be sure to enable the `collect_model_data` flag.

In [9]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.exceptions import WebserviceException

deployment_config = AksWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)
service_name = 'drift-aks-service'

service = Model.deploy(ws, service_name, [model], inference_config, deployment_config, aks_target)

service.wait_for_deployment(True)
print(service.state)

Running.....................................................................
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy


In [10]:
service.update(collect_model_data=True)

## Run recent weather data through the webservice 

The below cells take the past 2 days of weather data, filter and transform using the same processes as the training dataset, and runs the data through the service.

In [11]:
from datetime import datetime, timedelta
from azureml.opendatasets import NoaaIsdWeather

start = datetime.today() - timedelta(days=2)
end = datetime.today()

isd = NoaaIsdWeather(start, end)

df = isd.to_pandas_dataframe().fillna(0)
df = df[df['stationName'].str.contains('FLORIDA', regex=True, na=False)]

X_features = ['latitude', 'longitude', 'temperature', 'windAngle', 'windSpeed']
y_features = ['elevation']

X = df[X_features]
y = df[y_features]

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=10/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=10/part-00122-tid-5528606016710268819-0e3b2550-b4a0-4062-8f9c-d1294ec0c319-1751516.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=14049.79 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=14049.79 [ms]


In [12]:
import json

today_data = json.dumps({'data': X.values.tolist()})

data_encoded = bytes(today_data, encoding='utf8')
prediction = service.run(input_data=data_encoded)
print(prediction)

ERROR - Received bad response from service. More information can be found by calling `.get_logs()` on the webservice object.
Response Code: 503
Headers: {'x-ms-request-id': '3e1c1328-6c38-4823-8849-390f21d6724b', 'cache-control': 'max-age=0, private, must-revalidate', 'content-length': '60', 'date': 'Wed, 16 Oct 2019 05:44:14 GMT', 'server': 'Cowboy'}
Content: b'Too many requests for service drift-aks-service (overloaded)'



WebserviceException: WebserviceException:
	Message: Received bad response from service. More information can be found by calling `.get_logs()` on the webservice object.
Response Code: 503
Headers: {'x-ms-request-id': '3e1c1328-6c38-4823-8849-390f21d6724b', 'cache-control': 'max-age=0, private, must-revalidate', 'content-length': '60', 'date': 'Wed, 16 Oct 2019 05:44:14 GMT', 'server': 'Cowboy'}
Content: b'Too many requests for service drift-aks-service (overloaded)'
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Received bad response from service. More information can be found by calling `.get_logs()` on the webservice object.\nResponse Code: 503\nHeaders: {'x-ms-request-id': '3e1c1328-6c38-4823-8849-390f21d6724b', 'cache-control': 'max-age=0, private, must-revalidate', 'content-length': '60', 'date': 'Wed, 16 Oct 2019 05:44:14 GMT', 'server': 'Cowboy'}\nContent: b'Too many requests for service drift-aks-service (overloaded)'"
    }
}

## Create an Azure Machine Learning Compute cluster

The data drift capability needs a compute target for computing drift and other data metrics. 

In [13]:
from azureml.core.compute import AmlCompute, ComputeTarget

compute_name = 'cpu-cluster'

if compute_name in ws.compute_targets:
    compute_target = ws.compute_targets[compute_name]
    if compute_target and type(compute_target) is AmlCompute:
        print('found compute target. just use it. ' + compute_name)
else:
    print('creating a new compute target...')
    provisioning_config = AmlCompute.provisioning_configuration(vm_size='STANDARD_D3_V2', min_nodes=0, max_nodes=2)

    # create the cluster
    compute_target = ComputeTarget.create(ws, compute_name, provisioning_config)

    # can poll for a minimum number of nodes and for a specific timeout.
    # if no min node count is provided it will use the scale settings for the cluster
    compute_target.wait_for_completion(show_output=True, min_node_count=None, timeout_in_minutes=20)

    # For a more detailed view of current AmlCompute status, use get_status()
    print(compute_target.get_status().serialize())

creating a new compute target...
Creating
Succeeded
AmlCompute wait for completion finished
Minimum number of nodes requested have been provisioned
{'allocationStateTransitionTime': '2019-10-16T05:44:19.613000+00:00', 'scaleSettings': {'maxNodeCount': 2, 'nodeIdleTimeBeforeScaleDown': 'PT120S', 'minNodeCount': 0}, 'errors': None, 'vmPriority': 'Dedicated', 'currentNodeCount': 0, 'modifiedTime': '2019-10-16T05:44:32.904611+00:00', 'allocationState': 'Steady', 'nodeStateCounts': {'preemptedNodeCount': 0, 'leavingNodeCount': 0, 'unusableNodeCount': 0, 'runningNodeCount': 0, 'preparingNodeCount': 0, 'idleNodeCount': 0}, 'targetNodeCount': 0, 'creationTime': '2019-10-16T05:44:16.135735+00:00', 'provisioningStateTransitionTime': None, 'vmSize': 'STANDARD_D3_V2', 'provisioningState': 'Succeeded'}


## Wait 10 minutes

From the Model Data Collector, it can take up to (but usually less than) 10 minutes for data to arrive in your blob storage account. Wait 10 minutes to ensure cells below will run.

In [14]:
import time

time.sleep(600)

## Create and update the data drift object

In [15]:
from azureml.datadrift import DataDriftDetector, AlertConfiguration

services = [service_name]
start = datetime.now() - timedelta(days=2)
feature_list = X_features
alert_config = AlertConfiguration(['cody.peterson@microsoft.com'])

try:
    monitor = DataDriftDetector.create(ws, model.name, model.version, services, frequency='Day', 
                                       schedule_start=datetime.utcnow() + timedelta(days=1), 
                                       alert_config=alert_config, 
                                       compute_target_name='cpu-cluster')
except KeyError:
    monitor = DataDriftDetector.get(ws, model.name, model.version)
    
monitor

2019-10-16 05:54:39,573 - azureml.datadrift._logging._telemetry_logger.datadriftdetector.create - WARNING - Deprecated, please use create_from_model(). Will be removed in future releases. - activity_type:InternalCall activity_name:datadriftdetector.create telemetry_component_name:azureml.datadrift activity_id:f58c5a8e-9a51-4a24-ba13-84c662779962 sdk_version:0.1.0.6188706


{'_model_version': 1, '_name': None, '_interval': 1, '_drift_threshold': 0.2, '_schedule_id': None, '_state': 'Disabled', '_schedule_start': datetime.datetime(2019, 10, 17, 5, 54, 39, 573402, tzinfo=<FixedOffset '+00:00'>), '_alert_config': {'email_addresses': ['cody.peterson@microsoft.com']}, '_workspace': Workspace.create(name='notebooks', subscription_id='b0d50d34-4247-4cae-871a-6b1799907b8d', resource_group='aml_git_nb_6188706'), '_target_dataset_id': '00000000-0000-0000-0000-000000000000', '_model_name': 'elevation-regression-model.pkl', '_latency': 0, '_frequency': 'Day', '_type': 'ModelBased', '_id': 'c398d3e8-19df-46a4-86e9-248688ad6194', '_client': <azureml.datadrift._restclient.datadrift_client.DataDriftClient object at 0x00000196816EE550>, '_baseline_dataset_id': '00000000-0000-0000-0000-000000000000', '_feature_list': None, '_services': ['drift-aks-service'], '_logger': <azureml.datadrift._logging._telemetry_logger_context_adapter._TelemetryLoggerContextAdapter object at 0x

In [16]:
# create an AlertConfiguration
alert_config = AlertConfiguration(['cody.peterson@microsoft.com'])

# many monitor settings can be updated 
monitor = monitor.update(alert_config = alert_config)
monitor = monitor.update(drift_threshold = 0.1)

monitor

{'_model_version': 1, '_name': None, '_interval': 1, '_drift_threshold': 0.1, '_schedule_id': None, '_state': 'Disabled', '_schedule_start': datetime.datetime(2019, 10, 17, 5, 54, 39, 573402, tzinfo=<FixedOffset '+00:00'>), '_alert_config': {'email_addresses': ['cody.peterson@microsoft.com']}, '_workspace': Workspace.create(name='notebooks', subscription_id='b0d50d34-4247-4cae-871a-6b1799907b8d', resource_group='aml_git_nb_6188706'), '_target_dataset_id': '00000000-0000-0000-0000-000000000000', '_model_name': 'elevation-regression-model.pkl', '_latency': 0, '_frequency': 'Day', '_type': 'ModelBased', '_id': 'c398d3e8-19df-46a4-86e9-248688ad6194', '_client': <azureml.datadrift._restclient.datadrift_client.DataDriftClient object at 0x00000196816EE550>, '_baseline_dataset_id': '00000000-0000-0000-0000-000000000000', '_feature_list': None, '_services': ['drift-aks-service'], '_logger': <azureml.datadrift._logging._telemetry_logger_context_adapter._TelemetryLoggerContextAdapter object at 0x

## Run the monitor on today's scoring data

Perform a data drift run on the data sent to the service earlier in this notebook. If you set your email address in the alert configuration and the drift threshold <=0.1 you should recieve an email alert to drift from this run.

Wait for the run to complete before getting the results. 

In [17]:
now = datetime.utcnow()
target_date = datetime(now.year, now.month, now.day)
run = monitor.run(target_date, services, feature_list=feature_list, compute_target_name='cpu-cluster')

In [18]:
run.wait_for_completion(wait_post_processing=True); time.sleep(60)

## Get and view results and metrics

For enterprise workspaces, the UI in the Azure Machine Learning studio can be used. Otherwise, the metrics can be queried in Python and plotted. 

In [19]:
results, metrics = monitor.get_output(run_id=run.id)

2019-10-16 05:56:13,419 - azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector - ERROR -  Run history is empty, please accomplish at lease one run before retrieving outputs. - workspace_name:notebooks sdk_version:0.1.0.6188706 model_version:1 workspace_location:eastus workspace_id:1fe456b5-d55a-42bd-a9dc-44feff4501d2 model_name:elevation-regression-model.pkl telemetry_component_name:azureml.datadrift subscription_id:b0d50d34-4247-4cae-871a-6b1799907b8d dd_id:c398d3e8-19df-46a4-86e9-248688ad6194


FileNotFoundError:  Run history is empty, please accomplish at lease one run before retrieving outputs.

In [20]:
drift_plots = monitor.show()

2019-10-16 05:56:14,601 - azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector - ERROR -  Run history is empty, please accomplish at lease one run before retrieving outputs. - workspace_name:notebooks sdk_version:0.1.0.6188706 model_version:1 workspace_location:eastus workspace_id:1fe456b5-d55a-42bd-a9dc-44feff4501d2 model_name:elevation-regression-model.pkl telemetry_component_name:azureml.datadrift subscription_id:b0d50d34-4247-4cae-871a-6b1799907b8d dd_id:c398d3e8-19df-46a4-86e9-248688ad6194


FileNotFoundError:  Run history is empty, please accomplish at lease one run before retrieving outputs.

## Enable the monitor's pipeline schedule

Turn on a scheduled pipeline which will anlayze the serving dataset for drift. 

In [21]:
monitor.enable_schedule()

## Next steps

  * See [our documentation](https://aka.ms/datadrift/aks) or [Python SDK reference](https://docs.microsoft.com/python/api/overview/azure/ml/intro)
  * [Send requests or feedback](mailto:driftfeedback@microsoft.com) on data drift directly to the team
  * Please open issues with data drift here on GitHub or on StackOverflow if others are likely to run into the same issue